# 12 — Two-Stage PCA Exploration for Robust Mahalanobis\n",
    "\n",
    "**Goal (exploration only — no clustering in this notebook):**\n",
    "\n",
    "Starting from the **raw padded HOG features** (8,100 per image type), understand how many\n",
    "PCA components survive a two-stage compression and which threshold combinations are\n",
    "**safe** for Robust Mahalanobis.\n",
    "\n",
    "**Why Robust Mahalanobis has a dimensionality limit:**  \n",
    "The robust covariance estimator needs at least 3 observations per feature.  \n",
    "With 416 patients the safe limit is `416 / 3 ≈ 138` components.  \n",
    "A single PCA pass on raw HOG at ≥ 70% variance already exceeds this for most image types —\n",
    "hence the two-stage strategy.\n",
    "\n",
    "**Two-stage strategy:**\n",
    "1. **Stage 1** — compress each image type independently: 8,100 raw HOG → k₁ components\n",
    "2. **Stage 2** — concatenate the 5 reduced vectors and compress again: 5×k₁ → k₂ final\n",
    "\n",
    "Variance percentages at each stage are relative to the **raw HOG features**, giving\n",
    "a true picture of information retained.

## Imports

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

print("Imports successful.")

Imports successful.


## Step 1 — Load raw padded HOG features\n",
    "\n",
    "`raw_hog_features_per_image.csv` was built by notebook 01 with aspect-ratio padding.\n",
    "It contains **416 patients × 40,501 columns**:\n",
    "\n",
    "| Block | Columns | Range |\n",
    "|-------|---------|-------|\n",
    "| patient_id | 1 col | — |\n",
    "| cor HOG | 8,100 cols | `cor_HOG_0` … `cor_HOG_8099` |\n",
    "| gfc_sag HOG | 8,100 cols | `gfc_sag_HOG_0` … |\n",
    "| gfc_tra HOG | 8,100 cols | `gfc_tra_HOG_0` … |\n",
    "| masked_tra HOG | 8,100 cols | `masked_tra_HOG_0` … |\n",
    "| sbj_sag HOG | 8,100 cols | `sbj_sag_HOG_0` … |\n",
    "\n",
    "We load the file once and slice each image block by column name.

In [2]:
NB_FOLDER = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NB_FOLDER, '..', '..'))
DATA_PIPELINE_DIR = os.path.join(PROJECT_ROOT, 'notebooks', '01_data_pipeline')
OUTPUT_DIR = NB_FOLDER
DATA_DIR = PROJECT_ROOT         # legacy alias: used to glob OAS1_* raw patient folders
NB_DIR = DATA_PIPELINE_DIR      # legacy alias: shared pipeline inputs / outputsHOG_LEN     = 8_100
IMAGE_TYPES = ["cor", "gfc_sag", "gfc_tra", "masked_tra", "sbj_sag"]

print("Loading raw_hog_features_per_image.csv (may take a few seconds) ...")
df_hog = pd.read_csv(os.path.join(NB_DIR, "raw_hog_features_per_image.csv"))
print(f"Loaded: {df_hog.shape}")

# Slice each image type's 8,100-column block
hog_matrices = {}
for itype in IMAGE_TYPES:
    cols = [f"{itype}_HOG_{i}" for i in range(HOG_LEN)]
    X    = df_hog[cols].to_numpy(dtype=np.float64)
    hog_matrices[itype] = X
    print(f"  {itype:<14} shape={X.shape}")

N_PATIENTS     = next(iter(hog_matrices.values())).shape[0]
SAFE_THRESHOLD = N_PATIENTS / 3

print(f"\nPatients          : {N_PATIENTS}")
print(f"Mahalanobis limit : n_components < {SAFE_THRESHOLD:.2f}  (= {N_PATIENTS} / 3)")
print(f"SAFE if final n_components <= {int(SAFE_THRESHOLD)}")

Loading raw_hog_features_per_image.csv (may take a few seconds) ...
Loaded: (416, 40501)
  cor            shape=(416, 8100)
  gfc_sag        shape=(416, 8100)
  gfc_tra        shape=(416, 8100)
  masked_tra     shape=(416, 8100)
  sbj_sag        shape=(416, 8100)

Patients          : 416
Mahalanobis limit : n_components < 138.67  (= 416 / 3)
SAFE if final n_components <= 138


## Step 2 — Stage 1 PCA exploration

For each image type fit PCA on the raw 8,100 HOG features at five variance thresholds.
These percentages are relative to the **original HOG variance** — a true measure of
information retained.

Note: PCA is capped at `min(416 patients, 8100 features) = 416` components,
so 100% variance would need at most 416 components.

In [3]:
STAGE1_THRESHOLDS = [0.50, 0.60, 0.70, 0.80, 0.90]

stage1_components = {}   # itype → {threshold → n_components}

for itype, X in hog_matrices.items():
    print(f"\nImage: {itype}  (input shape: {X.shape}  —  raw HOG features)")
    print(f"  {'Variance threshold':<22} | {'N components':>13} | {'> safe limit?':>14}")
    print(f"  {'─'*55}")
    stage1_components[itype] = {}
    for thresh in STAGE1_THRESHOLDS:
        pca = PCA(n_components=thresh, random_state=42)
        pca.fit(X)
        n    = pca.n_components_
        warn = " ← EXCEEDS LIMIT" if n >= SAFE_THRESHOLD else ""
        stage1_components[itype][thresh] = n
        print(f"  {thresh*100:.0f}%{'':<20} | {n:>13}{warn}")


Image: cor  (input shape: (416, 8100)  —  raw HOG features)
  Variance threshold     |  N components |  > safe limit?
  ───────────────────────────────────────────────────────
  50%                     |            79
  60%                     |           114
  70%                     |           157 ← EXCEEDS LIMIT
  80%                     |           212 ← EXCEEDS LIMIT
  90%                     |           288 ← EXCEEDS LIMIT

Image: gfc_sag  (input shape: (416, 8100)  —  raw HOG features)
  Variance threshold     |  N components |  > safe limit?
  ───────────────────────────────────────────────────────
  50%                     |            70
  60%                     |           103
  70%                     |           144 ← EXCEEDS LIMIT
  80%                     |           199 ← EXCEEDS LIMIT
  90%                     |           277 ← EXCEEDS LIMIT

Image: gfc_tra  (input shape: (416, 8100)  —  raw HOG features)
  Variance threshold     |  N components |  > safe limit?
  ─

## Step 3 — Stage 2 PCA exploration

For each Stage 1 threshold (70%, 80%, 90%):
1. Compress each image type independently using Stage 1 PCA
2. Concatenate all 5 reduced matrices horizontally → merged matrix
3. Fit Stage 2 PCA on the merged matrix at four thresholds (70%, 80%, 90%, 95%)
4. Record the final component count

In [4]:
STAGE1_EXPLORE    = [0.50, 0.60, 0.70, 0.80, 0.90]
STAGE2_THRESHOLDS = [0.70, 0.80, 0.90, 0.95]

summary_rows = []

for s1_thresh in STAGE1_EXPLORE:
    # ── Stage 1: compress each image type from raw HOG ────────────────────────
    reduced_blocks = []
    n_per_type     = []
    for itype, X in hog_matrices.items():
        pca1  = PCA(n_components=s1_thresh, random_state=42)
        X_red = pca1.fit_transform(X)
        reduced_blocks.append(X_red)
        n_per_type.append(X_red.shape[1])

    X_merged    = np.concatenate(reduced_blocks, axis=1)
    merged_cols = X_merged.shape[1]

    print(f"\nStage 1 = {s1_thresh*100:.0f}%  "
          f"→ components per type: {n_per_type}  "
          f"→ merged shape: {X_merged.shape}")

    # ── Stage 2: compress merged matrix ──────────────────────────────────────
    row = {
        "stage1_threshold": f"{s1_thresh*100:.0f}%",
        "merged_columns":   merged_cols,
    }
    for s2_thresh in STAGE2_THRESHOLDS:
        pca2 = PCA(n_components=s2_thresh, random_state=42)
        pca2.fit(X_merged)
        n2   = pca2.n_components_
        row[f"stage2_{s2_thresh*100:.0f}pct"] = n2
        flag = "SAFE  " if n2 < SAFE_THRESHOLD else "UNSAFE"
        print(f"  Stage 2 = {s2_thresh*100:.0f}%  →  {n2:>4} final components  ({flag})")

    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print("\nSummary table ready.")


Stage 1 = 50%  → components per type: [79, 70, 71, 80, 21]  → merged shape: (416, 321)
  Stage 2 = 70%  →    56 final components  (SAFE  )
  Stage 2 = 80%  →    86 final components  (SAFE  )
  Stage 2 = 90%  →   133 final components  (SAFE  )
  Stage 2 = 95%  →   173 final components  (UNSAFE)

Stage 1 = 60%  → components per type: [114, 103, 105, 111, 45]  → merged shape: (416, 478)
  Stage 2 = 70%  →    81 final components  (SAFE  )
  Stage 2 = 80%  →   120 final components  (SAFE  )
  Stage 2 = 90%  →   178 final components  (UNSAFE)
  Stage 2 = 95%  →   227 final components  (UNSAFE)

Stage 1 = 70%  → components per type: [157, 144, 147, 150, 85]  → merged shape: (416, 683)
  Stage 2 = 70%  →   108 final components  (SAFE  )
  Stage 2 = 80%  →   154 final components  (UNSAFE)
  Stage 2 = 90%  →   220 final components  (UNSAFE)
  Stage 2 = 95%  →   273 final components  (UNSAFE)

Stage 1 = 80%  → components per type: [212, 199, 202, 200, 144]  → merged shape: (416, 957)
  Stage 2 =

## Step 4 — Flag safe combinations for Robust Mahalanobis

A combination is **SAFE** if the final Stage 2 component count is **< 416 / 3 ≈ 138.67**,
meaning the covariance matrix can be estimated reliably with 416 patients.
Combinations that exceed this are flagged **UNSAFE**.

In [5]:
stage2_cols = [f"stage2_{int(t*100)}pct" for t in STAGE2_THRESHOLDS]

print(f"{'Stage1':>8} | {'Merged':>8} | ", end="")
print(" | ".join(f"S2 {int(t*100)}%" for t in STAGE2_THRESHOLDS))
print("─" * 75)

for _, row in df_summary.iterrows():
    print(f"  {row['stage1_threshold']:>6} | {row['merged_columns']:>6} | ", end="")
    cells = []
    for col in stage2_cols:
        n    = int(row[col])
        flag = "SAFE  " if n < SAFE_THRESHOLD else "UNSAFE"
        cells.append(f"{n:>4} ({flag})")
    print(" | ".join(cells))

print(f"\nSafe limit: final components < {SAFE_THRESHOLD:.2f}  (= {N_PATIENTS} patients / 3)")

# ── Summary of recommended combinations ──────────────────────────────────────
print("\nRecommended combinations (SAFE for Robust Mahalanobis):")
print(f"  {'Stage 1':>8} | {'Stage 2':>8} | {'Final components':>18}")
print("  " + "─" * 42)
found = False
for _, row in df_summary.iterrows():
    for col, s2_t in zip(stage2_cols, STAGE2_THRESHOLDS):
        n = int(row[col])
        if n < SAFE_THRESHOLD:
            print(f"  {row['stage1_threshold']:>8} | {s2_t*100:.0f}%{'':<6} | {n:>18}")
            found = True
if not found:
    print("  None of the tested combinations are safe — try lower thresholds.")

  Stage1 |   Merged | S2 70% | S2 80% | S2 90% | S2 95%
───────────────────────────────────────────────────────────────────────────
     50% |    321 |   56 (SAFE  ) |   86 (SAFE  ) |  133 (SAFE  ) |  173 (UNSAFE)
     60% |    478 |   81 (SAFE  ) |  120 (SAFE  ) |  178 (UNSAFE) |  227 (UNSAFE)
     70% |    683 |  108 (SAFE  ) |  154 (UNSAFE) |  220 (UNSAFE) |  273 (UNSAFE)
     80% |    957 |  135 (SAFE  ) |  186 (UNSAFE) |  257 (UNSAFE) |  310 (UNSAFE)
     90% |   1348 |  161 (UNSAFE) |  217 (UNSAFE) |  289 (UNSAFE) |  338 (UNSAFE)

Safe limit: final components < 138.67  (= 416 patients / 3)

Recommended combinations (SAFE for Robust Mahalanobis):
   Stage 1 |  Stage 2 |   Final components
  ──────────────────────────────────────────
       50% | 70%       |                 56
       50% | 80%       |                 86
       50% | 90%       |                133
       60% | 70%       |                 81
       60% | 80%       |                120
       70% | 70%       |        